# Week 20: Computer Vision, 3D Vision & NeRFs
## Notebook 08: Depth Anything: Foundation Models for Monocular Depth

### 1. Conceptual Intuition & Theoretical Background
Monocular depth estimation (predicting 3D scene depth $d$ from a single 2D RGB image) is inherently ill-posed because infinitely many 3D physical geometries can produce the exact same 2D projection.

Moreover, combining diverse datasets (LiDAR scans, stereo disparities, synthetic CGI) is difficult because sensors have **incompatible depth scales and unknown camera baselines**.

**Depth Anything** resolves this using:
1. **Scale-and-Shift Invariant Loss (SSI):**
Normalizes predicted depth $\hat{d}$ and ground-truth depth $d^*$ before computing loss:
$$\hat{d}^* = \frac{\hat{d} - t(\hat{d})}{s(\hat{d})}, \quad d^* = \frac{d - t(d)}{s(d)}$$
Where $t(d) = \text{median}(d)$ and $s(d) = \frac{1}{N} \sum |d - t(d)|$.
$$\mathcal{L}_{\text{SSI}} = \frac{1}{N} \sum_{i=1}^{N} |\hat{d}_i^* - d_i^*|$$

2. **Large-Scale Unlabeled Distillation:**
Trains a teacher model on annotated datasets, then pseudo-labels 62 million unlabeled web images with strong data perturbations, producing fine depth boundaries.

---

### 2. Scratch Python Implementation: Scale-and-Shift Invariant Loss

In [ ]:
import numpy as np

def normalize_scale_shift(depth_map):
    """
    Normalizes depth map using median and mean absolute deviation:
    d_norm = (d - median(d)) / (mean(|d - median(d)|) + eps)
    """
    t = np.median(depth_map)
    s = np.mean(np.abs(depth_map - t)) + 1e-6
    return (depth_map - t) / s

def scale_and_shift_invariant_loss(pred_depth, true_depth):
    """
    Computes SSI L1 loss between two depth maps with different arbitrary scales.
    """
    pred_norm = normalize_scale_shift(pred_depth)
    true_norm = normalize_scale_shift(true_depth)
    return np.mean(np.abs(pred_norm - true_norm))

# Generate Synthetic Ground-Truth Depth Map (Linear Depth Gradient)
H, W = 32, 32
y_coords, x_coords = np.mgrid[0:H, 0:W]
true_depth = (y_coords * 0.2 + x_coords * 0.1) + 2.0  # Range ~ [2.0, 11.6]

# Scenario: Predictor outputs depth with an unknown arbitrary scale (x 50) and shift (+ 200)
pred_distorted = true_depth * 50.0 + 200.0 + np.random.normal(0, 0.5, (H, W))

# Standard MSE explodes due to scale mismatch
naive_mse = np.mean((pred_distorted - true_depth) ** 2)

# SSI Loss resolves scale and shift distortion
ssi_error = scale_and_shift_invariant_loss(pred_distorted, true_depth)

print("--- Scale-and-Shift Invariant (SSI) Depth Loss ---")
print(f"True Depth Range:               [{true_depth.min():.2f}, {true_depth.max():.2f}]")
print(f"Distorted Prediction Range:     [{pred_distorted.min():.2f}, {pred_distorted.max():.2f}]")
print(f"Naive MSE Loss:                 {naive_mse:.2f} (Completely distorted by scale)")
print(f"Scale-Shift Invariant Loss:     {ssi_error:.4f} (Accurately captures relative geometric depth!)")

### 3. Step-by-Step Code Walkthrough
1. **Median Normalization**: Aligns baseline shifts ($t$) without outlier sensitivity.
2. **Mean Absolute Deviation (MAD)**: Normalizes dynamic scale ($s$).
3. **Sensor-Agnostic Training**: Allows joint training on synthetic data and raw LiDAR.

---

### 4. Industry Applications & Trade-offs
- **Autonomous Driving & 3D Reconstruction:** Generating instant 3D point clouds from monocular dashboard cameras.
- **Trade-off:** Produces relative depth rather than metric depth without sensor calibration.

---

### 5. Top Interview Questions & Answers
**Q1: Why is monocular depth estimation scale-ambiguous?**
> **Answer:** A small object close to the camera projects the exact same image pixels as a large object far away. Without multiple cameras or sensor baselines, absolute physical scale cannot be uniquely determined from a single RGB image.

**Q2: How does the Scale-and-Shift Invariant loss enable training across heterogeneous depth datasets?**
> **Answer:** By normalizing each depth map by its median and mean absolute deviation, it aligns disparate depth scales (such as LiDAR meters vs stereo disparity values) into a consistent coordinate space.